tech0 wek4の課題
大枠はstep0~step5までの6ステップがある。
・その中で、step0~4までが個人活動、
・step5~6がチーム活動である

##Step0:UIの設計
完成図形をイメージする

書くものは、以下の3つ
1.検索結果の画面
2.統計ダッシュボードの画面
3.タブのラベルと、アイコン

v1.0の完成イメージ

Tec Search V1.0
Projecto Zero --社内ナレッジ検索エンジン（TF - IDF ランキング搭載）

ーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーー
[ 検索 🔍 ] [ クローラー ][ 一覧 ] 
ーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーー

キーワードを入力する
[----- ]  

検索結果：⚪︎⚪︎件
---------------------------------------------------------------
 DX戦略レポート
 　┗　デジタルトランスフォーメーションがXXYYZZ
　TTL SCORE 87.2
---------------------------------------------------------------


---------------------------------------------------------------


---------------------------------------------------------------

Step1
1_1.大前提として、W4で変えるのは「並べ方」だけ
今までは、出現回数順に並べていた。
それを、関連度で「賢く」並べるのが、今週の主役

1_2.並べるを担う新しいファイルとして、ranking.pyのファイルを紐付ける。

v1.0のファイルは
tech0-search-v1.0
 ┗ app.py               ← メインアプリ（Streamlit）
 ┗ ranking.py           ← SearchEngine クラス（TF-IDFエンジン）
 ┗ database.py          ← DB操作の一元管理
 ┗ schema.sql           ← テーブル定義（SQLをファイルに分離）
 ┗ crawler/py           ← W2から終了（クロール）
 ┗ page_w2.json         ← W2で集めたデータ（DBへの移行先）
 ┗ requirements.txt     ← 依存ライブラリ？
 ┗ data/
   ┗ tech0_search.db    ←SQLite DB（アプリが自動生成）



Step2 ： TF-IDFとは何か？
このStepが終わったら、
TF = 出現回数 + 総単語数を手計算で出せる
IDF = 「その単語の珍しさ」で、「の」「は」はほぼ0になると説明できる
現状のmatch_countでは何が困るのか、TF-IDFがそれをどう解決するのか言える

2_1.TF（単語の出現回数）とは？
・あるページ内で、その単語が何回出てくるか？を割合で表したもの
・例えば「DX推進で営業店の効率が向上 DXは重要な戦略」の場合は
　DX 推進 で 営業店 の 効率 が 向上 DX は 重要 な 戦略 = 13単語
　13単語のうち、DXは2回出てくる= 2/13 = 0.154
　ただし、一般的な言葉にした時に、「の」などの接続詞は多くなる　→そこで、IDFが登場する

ここで確認してほしいこと
・words.count("DX")で単語の出現回数を数えられる
・TFの計算式「出現回数　÷ 総単語数」が実際のコードと一致
・「の」は、「DX」の半分したTFがないことが確認できる

この後本格的な説明になるが、その前に、pythonでscikit-learnをインストールする↓
# scikit-learn のインストール（TF-IDF実装に必須）
# ターミナルで実行してもOK： pip install scikit-learn


In [1]:
pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 57.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 61.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [9]:
##TFを手計算で確認する

##サンプルテキスト
##日本語を単語分割するには、「形態素解析ライブラリ」が必要。語義を簡素化するために、英語のようにトークン別にスペースを儲ける
text = "DX 推進 で 営業店 の 効率 が 向上 DX は 重要 な 戦略"
words = text.split()

##単語を数える
word_count = len(words)
print(f"総単語:{word_count}語")

#出現回数を数える
dx_count = words.count("DX")
no_count = words.count("の")
print(f"「DX」の出現回数：{dx_count}回")
print(f"「の」の出現回数：{no_count}回")

##TFを計算する
tf_dx = dx_count / word_count
tf_no = no_count / word_count
print(f"TF(DX) : {tf_dx:.3f}")
print(f"TF(の) : {tf_no:.3f}")


総単語:13語
「DX」の出現回数：2回
「の」の出現回数：1回
TF(DX) : 0.154
TF(の) : 0.077


2_2 IDF（逆文書頻度）とは
・IDF = Inverse Document Frequency
・その単語が、全体の何割のページに出てくるか？の逆数

例えば、「の」「は」「です」と言う言葉は、ほぼ全ページに出てくるのでIDFが低い。
「メダリオン」「TF-IDF」などは、全体の一部のページにしか出てこないのでIDFが高い
つまり、単語の珍しさを数値化する指標

例えば以下pythonで試してみると、以下のことがわかる
・「の」のIDFがほぼ0に近い
・「メダリオン」のIDが「DX」より高い
・match.log()は自然大数を計算する関数である

In [11]:
import math

#IDFを手計算で確認する

#全ページ数
total_pages = 30

#各単語が含まれるページ数
page_with_dx = 8
page_with_no = 29
page_with_medallion = 2

#IDFを計算する
idf_dx = math.log(total_pages / page_with_dx)
idf_no = math.log(total_pages / page_with_no)
idf_medallion = math.log(total_pages / page_with_medallion)

print(f"IDF(DX)      :{idf_dx:.3f})")
print(f"IDF(の)      :{idf_no:.3f})")
print(f"IDF(メダリオン)      :{idf_medallion:.3f})")

IDF(DX)      :1.322)
IDF(の)      :0.034)
IDF(メダリオン)      :2.708)


2_3:TF × ÎDF = TF-IDFスコア
先ほどの数値だと

単語    TF      IDF     TDーIDF
DX      0.154   1.322   0.203
の      0.077   0.034   0.003

「DX」のスコアは0.2,「の」のスコアは0.003
概念的には、「よく出てくる」かつ「珍しい」単語のスコアが上がる

In [13]:
#TF-IDFスコアを手計算で求める

import math

#先ほど計算した値を使う
tf_dx = 2 / 13      #TF(DX)
tf_no = 1 / 13      #TF(の)

idf_dx = math.log( 30 / 8 )
idf_no = math.log( 30 / 29 )

#TF-IDFを計算する
tfidf_dx = tf_dx * idf_dx
tfidf_no = tf_no * idf_no

print(f"TF-IDF(DX) :{tfidf_dx:.3f}")
print(f"TF-IDF(の) :{tfidf_no:.3f}")
print()
print(f"「DX」は「の」の {tfidf_dx / tfidf_no:.0f}倍のスコアになった")

TF-IDF(DX) :0.203
TF-IDF(の) :0.003

「DX」は「の」の 78倍のスコアになった


Step3:コサイン類似度について
・TF-IDFは、単語1つの重要度が知れる
・一方、検索クエリの文章と、ページ文章の近さは測れない
・これを測るためには、ベクトル化とコサイン類似度を理解する必要がある

Step3で理解するものは、ベクトル化とコサイン類似度の話

3-1.ベクトル化とは
・ベクトル化とは、文章を数値の配列に変換すること
・コンピュータで比較するには、「数値」に変換する必要がある
・TF-IDFで各単語が数値にあったものを、文章の数値配列で数値化させる


| 単語 | DX | 推進 | 営業店 | AI | 機械学習 |
|------|----|------|--------|----|----------|
| 文章A（DX推進...） | **0.7** | **0.5** | **0.6** | 0.0 | 0.0 |
| 文章B（AI機械...） | 0.0 | 0.0 | 0.0 | **0.8** | **0.9** |
| クエリ「DX」 | **1.0** | 0.0 | 0.0 | 0.0 | 0.0 |

以下のPythonで確認すること
・文章が「単語　× スコア」の数値配列（ベクトル）に変換されている
・クエリも同じ語彙を使ってベクトル化されている
・クエリに含まれない単語のスコアは0になっている

In [18]:
#ベクトル化の仕組みを確認する
from sklearn.feature_extraction.text import TfidfVectorizer

#サンプルテキスト
docs =  [
    "DX 推進 営業店 効率 向上",      # 文章A
    "AI 機械学習 業務 効率 化",       # 文章B
    "DX 営業店 IoT センサー コスト",  # 文章C
]
query = "DX 営業店"

#TF-IDFでベクトル化する
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(docs)

#クエリもベクトル化する
query_vec = vectorizer.transform([query])

#語彙（インデックス化された単語）を確認する
feature_names = vectorizer.get_feature_names_out()
print("語彙：",feature_names)
print()

#クエリのベクトル（各単語のTF-IDFスコア）を確認
print("クエリ「DX 営業店」のベクトル（スコアが 0 ではない単語）：")
for word, score in zip(feature_names, query_vec.toarray()[0]):
    if score > 0:
        print(f"  '{word}': {score:.3f}")


語彙： ['ai' 'dx' 'iot' 'コスト' 'センサー' '効率' '向上' '営業店' '推進' '業務' '機械学習']

クエリ「DX 営業店」のベクトル（スコアが 0 ではない単語）：
  'dx': 0.707
  '営業店': 0.707


3-2.コサイン類似度とは
・コサイン類似度 = 2つのベクトルの「向き」の近さを0~1で表す
・ベクトルを「矢印」と考えると、2つの矢印が同じ方向を向いているほど、「似ている」という判断になる
・コサインを使う理由は、文章の長さに影響されない
・完全一致じゃなくても、「似ている度合い」が測れる

以下のpythonで確認すること
・クエリ「DX 営業店」に対して文章Aと文章Cが高いスコアになっている
・文章Bはスコアが0に近い（「AI」「機械学習」はクエリに含まないため）
・cosine_similarity(query_vec,tfidf_matrix)[0]の[0]は
　「1件のクエリに対する全文章のスコア配列」を取り出している


In [19]:
#コサイン類似度の計算
from sklearn.metrics.pairwise import cosine_similarity

#3-1で作ったベクトルをそのまま使う
similarities = cosine_similarity(query_vec, tfidf_matrix)[0]

print("クエリ「DX 営業店」との類似度：")
labels = [
    "文章A（DX推進・営業店）",
    "文章B（AI・機械学習）",
    "文章C（DX・営業店・IoT）",
]
for label, score in zip(labels,similarities):
    bar = "█" * int(score * 20)
    print(f"    {score:.3f}    {bar}    {label}")

print()
best_idx = similarities.argmax()
print(f"→ 最もクエリに近い文章：{labels[best_idx]}(スコア：{similarities[best_idx]:.3f})")

クエリ「DX 営業店」との類似度：
    0.557    ███████████    文章A（DX推進・営業店）
    0.000        文章B（AI・機械学習）
    0.528    ██████████    文章C（DX・営業店・IoT）

→ 最もクエリに近い文章：文章A（DX推進・営業店）(スコア：0.557)


In [22]:
# ── 演習 3-A 解答欄 ──
# 4つの文章に対して「AI 機械学習」でコサイン類似度を計算しよう
# さらに query を "DX IoT" に変えると順位がどう変わるかも確認しよう

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "DX 推進 営業店 効率 向上",
    "AI 機械学習 業務 効率 化",
    "Python データ 分析 AI モデル 作成",
    "IoT センサー 店舗 DX コスト 削減",
]

# ここに書いてみよう ↓
#TF-IDFでベクトル化する
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(docs)

def rank(query):
    query_vec = vectorizer.transform([query])
    #クエリも同じ語彙でベクトル化
    sims = cosine_similarity(query_vec, tfidf_matrix)[0]

    print(f"クエリ「{query}」")
    for i in sorted(range(len(docs)), key=lambda k:sims[k],reverse=True):
        bar = "█" * int(sims[i] * 20)
        print(f" {sims[i]:.3f} {bar:<20} {docs[i]}")
    print()

rank("AI 機械学習")
rank("DX IoT")


クエリ「AI 機械学習」
 0.707 ██████████████       AI 機械学習 業務 効率 化
 0.206 ████                 Python データ 分析 AI モデル 作成
 0.000                      DX 推進 営業店 効率 向上
 0.000                      IoT センサー 店舗 DX コスト 削減

クエリ「DX IoT」
 0.537 ██████████           IoT センサー 店舗 DX コスト 削減
 0.237 ████                 DX 推進 営業店 効率 向上
 0.000                      AI 機械学習 業務 効率 化
 0.000                      Python データ 分析 AI モデル 作成



Step4:SearchEngineクラスの実装

Step2、3をコードに落とす
手実装　→ scikit-learn → クラス化の順に進むことで、エラーが出た時に原因をおえる状態を目指す

ここでranking.pyを触る（作る）

▼仕組みを知る
・Step2では、TF-IDFを理解した
・Step3では、ベクトル化とコサイン類似度を学んだ
・Step4では、次の順序で実装を進める

1. calc_tf() / calc_idf を自分で記載する（step2~3の復習）
2. scikit-learnのTfidVectorizerを使う（ライブラリ版）
3. cosine_similarityで文書の近さを測る
4. SearchEngineクラスにまとめる

In [24]:
#TF(単語の出現頻度)を計算する関数

def calc_tf(text: str, word: str) -> float:
    """
    1つのテキスト内での単語のTF(出現頻度)を計算する

    Args:
        text : 対象テキスト（スペース区切り）
        word : 調べたい単語

    Returns:
        TFスコア（0~1 の少数）
    """
    words = text.split()        #テキストを単語リストに分解

    if len(words) == 0:
        return 0.0              #空のテキストは 0 を返す

    count = words.count(word)   #対象単語の出現回数を数える
    return count / len(words)   #出現回数 ÷ 総単語数 = TF

#動作確認
text1 = "DX 推進 で 営業店 の 効率 が 向上 DX は 重要 な 戦略"
print(f"calc_tf(text1,'DX') : {calc_tf(text1,'DX'):.3f}")
print(f"calc_tf(text1,'の') : {calc_tf(text1,'の'):.3f}")

calc_tf(text1,'DX') : 0.154
calc_tf(text1,'の') : 0.077


以下のPythonで確認すること
・math.logで対数を計算できる。すなわち、IDFを計算している
・出現ページ数が多いほど、IDFが低くなる傾向が確認できる

In [26]:
#calc_idf関数を書いてみる

import math

#IDF（逆文書頻度）を計算する関数

def calc_idf(docs: list , word: str) -> float:
    """
    複数のドキュメント（ページ）内での単語のIDFを計算する

    Args:
        docs : テキストのリスト（各ページのテキスト）
        word：調べたい単語
    
    Returns:
        IDF スコア（0以上の少数）
    """

    #その単語が含まれるページ数を数える
    pages_with_word = sum(1 for doc in docs if word in doc.split())

    if pages_with_word == 0:
        return 0.0              #どのページにも出てこない単語は0を返す

    #IDFを計算する：log（総ページ数 / 含むページ数）
    return math.log(len(docs) / pages_with_word)


##動作確認
#日本語を単語分割するには、形態素解析ライブラリが必要。講義を簡素化するために英語のようにトークン別にスペースを儲けた
docs = [
    "DX 推進 で 営業店 の 効率 が 向上",
    "AI 活用 で 業務 効率 化 を 進める",
    "IoT センサー で 店舗 DX を 実現",
    "Python で AI モデル を 作る",
    "データ 分析 で 経営 判断 を 支援",
]

print(f"IDF(DX)     :{calc_idf(docs, 'DX'):.3f}")  #2ページに出現　→ 中程度
print(f"IDF(の)    :{calc_idf(docs,'の'):.3f}")    #1ページのみ　→ やや高い
print(f"IDF(効率)    :{calc_idf(docs,'効率'):.3f}")    #2ページに出現　→ 中程度
print(f"IDF(AI)    :{calc_idf(docs,'AI'):.3f}")    #2ページに出現　→ 中程度

IDF(DX)     :0.916
IDF(の)    :1.609
IDF(効率)    :0.916
IDF(AI)    :0.916


In [29]:
#TF-IDFスコアで検索ランキングを作る

def search_tfidf_manual(query: str, docs: list) -> list:
    """
    TF-IDFスコアでドキュメントをランキングする（手実装版）。

    Args:
        query   : 検索クエリ（スペース区切りで複数単語可
        docs    : テキストのリスト

    Returns:
        スコア降順のランキングリスト
    """
    results = []

    for i, doc in enumerate(docs):
        #クエリを単語に分析する
        query_words = query.split()

        #クエリの各単語について、TF-IDFスコアを計算して合計する
        score = 0.0
        for word in query_words:
            tf = calc_tf(doc, word)
            idf = calc_idf(docs, word)
            score += tf * idf       #TF * IDFを積み上げる

        if score > 0:
            results.append({"index": i,"score":round(score,4),"text":doc[:40]})

    #スコアの高い順に並べる
    results.sort(key=lambda x:x["score"], reverse=True)
    return results

#動作確認：「DX 効率」で検索
docs_pages = [
    "DX 推進 で 営業店 の 効率 が 向上 DX は 重要 な 戦略",
    "AI 活用 で 業務 効率 化 を 進める ツール の 導入",
    "IoT センサー で 店舗 DX を 実現 コスト 削減",
    "Python で AI モデル を 作る データ サイエンス 入門",
    "データ 分析 で 経営 判断 を 支援 BI ツール 活用",
]

results = search_tfidf_manual("DX 効率", docs_pages)
print("「DX 効率」での検索結果：")
for r in results:
    print(f" スコア{r['score']:.4f}:{r['text']}...")
    

「DX 効率」での検索結果：
 スコア0.2115:DX 推進 で 営業店 の 効率 が 向上 DX は 重要 な 戦略...
 スコア0.1018:IoT センサー で 店舗 DX を 実現 コスト 削減...
 スコア0.0833:AI 活用 で 業務 効率 化 を 進める ツール の 導入...


scikit-learnのTfidsVectorizerを使用
手実装で仕組みがわかった。次はライブラリを使用して、より高精度に実装する
ライブラリだと
・スペース区切りでなくても単語を分解できる
・バイグラム（2単語の組み合わせ）も考慮できる
・大量ページも高速に処理できる

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# TfidfVectorizerでインデックスを構築
# 日本語を単語分割するには「形態素解析ライブラリ」が必要。講義を簡素化するために英語のようにトークン別にスペースを設けた

docs_pages = [
    "DX 推進 で 営業店 の 効率 が 向上 DX は 重要 な 戦略",
    "AI 活用 で 業務 効率 化 を 進める ツール の 導入",
    "IoT センサー で 店舗 DX を 実現 コスト 削減",
    "Python で AI モデル を 作る データ サイエンス 入門",
    "データ 分析 で 経営 判断 を 支援 BI ツール 活用",
]

#TF=IDF ベクトライザーを生成
vectorizer = TfidfVectorizer(
    max_features=5000,      #語彙の最大数
    ngram_range=(1,2),      #ユニグラム（1語）とバイグラム（2語）の組み合わせ
    min_df=1,               #最低1ページに出てくる単語のみ
    max_df=0.95,            #95%以上のページに出てくる単語は除外
    sublinear_tf=True        #TFの対数スケーリング（お大きすぎる値を抑える）
)

#全ページのテキストをベクトル化する（fit_transform)
tfidf_matrix = vectorizer.fit_transform(docs_pages)
print(f"マトリックスの形状：{tfidf_matrix.shape}")
print(f"語彙数:{len(vectorizer.get_feature_names_out())} 語")

#検索クエリをベクトル化して類似度を計算する（transformのみ）
query = "DX 効率"
query_vec = vectorizer.transform([query])
scores = cosine_similarity(query_vec,tfidf_matrix).flatten()

print()
print("「DX 効率」のコサイン類似度スコア：")
for i,score in enumerate(scores):
    if score > 0:
        print(f" ページ{i+1}(スコア{score:.4f}) :{docs_pages[i][:40]}...")

マトリックスの形状：(5, 62)
語彙数:62 語

「DX 効率」のコサイン類似度スコア：
 ページ1(スコア0.4032) :DX 推進 で 営業店 の 効率 が 向上 DX は 重要 な 戦略...
 ページ2(スコア0.1675) :AI 活用 で 業務 効率 化 を 進める ツール の 導入...
 ページ3(スコア0.1604) :IoT センサー で 店舗 DX を 実現 コスト 削減...


#手実装をライブラリの結果を比べる
・順位の傾向が似ている（どちらもDXを含むページが上位）
・ライブラリ版の方が、バイグラムを考慮するため微妙にスコアが異なる
・cosine_similarityの仕組みはStep3で詳しく学んだ
・コサイン類似度は文章の長さに左右されない　→ 長短のページを公平に比較できる


4_5.Search Engineクラスに発展させる
#関数をクラスにまとめると、以下のメリットがある
・インデックス（TF-IDFマトリックス）を一度だけ使えば使いまわせる
・build_index() →search()という自然な使い方ができる
・チームで分担しやすい（ランキング担当が管理するファイルになる）

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import math
from datetime import datetime

class SearchEngine:
    """TF-IDFベースの検索エンジン（ranking.pyの本体"""

    def __init__(self):
        #TF-IDFのベクトライザーを初期化
        #日本語は単語の間にスペースがないため、既存の「単語区切り」では
        #「営業店DX推進レポート」が丸ごと1語になり、「DX」で検索をかけても一致しない
        #そこで、analyzer = "char_wb" （文字N-gram）で2~3文字のまとまりを特徴量にする
        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb",     #文字N-gram（日本語のまま使えるようにする）
            ngram_range=(2,3),      #2~3文字のまとまり
            max_features=5000,
            min_df=1,
            max_df=0.95,
            sublinear_tf=True       #TFの対数スケーリング
        )
        self.tfidf_matrix = None    #インデックス（後で構築）
        self.pages= []              #元のページデータを保持
        self.is_fitted = False      #インデックスが構築済みがのフラグ

    def build_index(self, pages:list):
        """
        全ページのTF-IDFインデックスを構築

        Args:
            pages:ページ情報の辞書リスト
        """
        if not pages:
            return

        self.pages = pages

        #各ページの「検索対象テキスト」を組み立て
        #タイトル・説明・キーワードに重みをつけるため、文字列を繰り返す
        corpus = []
        for p in pages:
            #keywordsがカンマ区切りの文字列の場合はリストに追加
            kw = p.get("keywords", "") or ""
            if isinstance(kw, str):
                kw_list = [k.strip() for k in kw.split(",") if k.strip()]
            else:
                kw_list = kw

            #重みづけを実施。タイトルは3倍、説明は2倍、キーワードは2倍の重み
            text = " ".join([
                (p.get("title","") + " ") * 3,          #タイトルは3倍
                (p.get("description","") + " ") * 2,    #説明は2倍
                (p.get("full_text","") + " ") * 1,      #本文は1倍
                (" ".join(kw_list) + " ") * 2,          #キーワードは2倍
            ])
            corpus.append(text)

        #YF-IDFマトリックスを構築
        self.tfidf_matrix = self.vectorizer.fit_transform(corpus)
        self.is_fitted = True
        print(f"インデックス構築完了 :{len(pages)}ページ")

    def search(self,query: str,top_n: int = 20) -> list:
        """
        TF-IDFベースの検索を実行
        
        Args:
            query : 検索クエリ
            top_n : 返す結果の最大値
        
        Returns:
            スコア付きの検索結果リスト
        """
        if not self.is_fitted or not query.strip():
            return[]

        #クエリをベクトル化してコサイン炊事どを計算する（Step3で学んだ内容）
        query_vec = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vec,self.tfidf_matrix)[0]

        #閾値以上のページだけ結果に含める
        results = []
        for idx, base_score in enumerate(similarities):
            if base_score > 0.01:
                page = self.pages[idx].copy()

                #追加スコアりんぐで最終スコアを計算
                final_score = self._calculate_final_score(page, base_score, query)

                #スコアをパーセント表示用に変換する
                page["relevance_score"] = round(float(final_score) *100, 1)
                page["base_score"] = round(float(base_score) * 100, 1)
                results.append(page)

        #スコアの高い順に並べて　top_n 件を返す
        results.sort(key=lambda x: x["relevance_score"],reverse=True)
        return results[:top_n]

    def _calculate_final_score(self, page: dict, base_score: float,query: str) -> float:
        """
        複数の要素を組み合わせて最終スコアを計算する（内部メソッド）
        Args：
            page: ページ情報
            base_score: TF-IDFベーススコア
            query: 検索クエリ
        
        Returns:
            最終スコア
        """

        score = base_score
        query_lower = query.lower()

        #1.タイトルマッチボーナス
        title = page.get("title", "").lower()
        if query_lower == title:
            score *=  1.8       #完全一致の場合、＋80%にする
        elif query_lower in title:
            score *= 1.4    #部分一致の場合、＋40%

        #2.キーワードマッチボーナス
        keywords = page.get("keywords",[])

        if isinstance(keywords, str):
            keywords = keywords.split(",")

        keywords_lower = [k.strip().lower() for k in keywords]

        if query_lower in keywords_lower:
            score *= 1.3    #キーワード数：＋30%

        #3.新鮮度ボーナス（90日以内おページは最大　＋20%）
        crawled_at = page.get("crawled_at","")
        if crawled_at:
            try:
                crawled = datetime.fromisoformat(crawled_at.replace("Z","+00:00"))
                days_old = (datetime.now() - crawled.replace(tzinfo=None)).days
                if days_old <= 90:
                    recency_bonus = 1 + (0.2 * (90 - days_old) / 90)
                    score *= recency_bonus
            except Exception:
                pass

        #4.文字数による調査
        word_count = page.get("word_count" ,0)
        if word_count < 50:
            score *= 0.7        #短すぎるページは70%に減点
        elif word_count > 10000:
            score *= 0.85       #長すぎるページは85%に減点

        return score

#---動作確認----------------------------------

#テスト用ページのデータ
sample_pages = [
    {"title": "DX戦略レポート2024", "description": "営業店のDX推進について解説",
     "full_text": "DX 推進 で 営業店 の 効率 が 向上 DX は 重要 な 戦略 デジタル 変革",
     "keywords": "DX,営業店,デジタル変革", "word_count": 50,
     "crawled_at": "2024-01-15T10:00:00"},
    {"title": "AI活用ガイド", "description": "業務効率化のためのAI導入",
     "full_text": "AI 活用 で 業務 効率 化 を 進める ツール の 導入 機械 学習",
     "keywords": "AI,機械学習,業務効率化", "word_count": 45,
     "crawled_at": "2024-02-20T15:00:00"},
    {"title": "IoTセンサー活用事例", "description": "店舗のIoT化事例",
     "full_text": "IoT センサー で 店舗 DX を 実現 コスト 削減 リアルタイム モニタリング",
     "keywords": "IoT,店舗,DX,センサー", "word_count": 60,
     "crawled_at": "2024-03-01T09:00:00"},
]

#SearchEngineを使ってみる
engine = SearchEngine()
engine.build_index(sample_pages)

# クエリは「 DX」」1語にする
# 複数語（例：「DX 営業店」だと、タイトル・キーワードのボーナス判定が
# クエリの文字列そのままの一致を見るため発火せず、2種類のスコアが同じ値になる
results = engine.search("DX")
print("\n「DX」での検索結果：")
for i,r in enumerate(results, 1):
    medal =  ["🥇", "🥈", "🥉"][i-1] if i <= 3 else str(i)
    print(f" {medal}{r['title']}(relevance:{r['relevance_score']} / base:{r['base_score']})")
        

インデックス構築完了 :3ページ

「DX」での検索結果：
 🥇DX戦略レポート2024(relevance:48.0 / base:26.4)
 🥈IoTセンサー活用事例(relevance:27.4 / base:21.1)


ここで確認すべきこと
・build_index()でインデックスを一回作れば、search()で何度でも使いまわせる
・relevance_score（最終スコア）と、base_score（TF-IDFの元スコア）の2種類がある
・タイトルに「DX」が含まれるページが高いスコアになる
・スコアが0.01未満のページは除外し、無関係なページが表示される問題を防いでいる

In [39]:
#演習問題
#検索クエリを変更して、ランキングがどう変わるかを試す
#　・AI 機械学習で検索したとき、1位はどのページになるか？
#　・IoTで検索したとき、スコアはどう変わるか？

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import math
from datetime import datetime

class SearchEngine:
    """TF-IDFベースの検索エンジン（ranking.pyの本体"""

    def __init__(self):
        #TF-IDFのベクトライザーを初期化
        #日本語は単語の間にスペースがないため、既存の「単語区切り」では
        #「営業店DX推進レポート」が丸ごと1語になり、「DX」で検索をかけても一致しない
        #そこで、analyzer = "char_wb" （文字N-gram）で2~3文字のまとまりを特徴量にする
        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb",     #文字N-gram（日本語のまま使えるようにする）
            ngram_range=(2,3),      #2~3文字のまとまり
            max_features=5000,
            min_df=1,
            max_df=0.95,
            sublinear_tf=True       #TFの対数スケーリング
        )
        self.tfidf_matrix = None    #インデックス（後で構築）
        self.pages= []              #元のページデータを保持
        self.is_fitted = False      #インデックスが構築済みがのフラグ

    def build_index(self, pages:list):
        """
        全ページのTF-IDFインデックスを構築

        Args:
            pages:ページ情報の辞書リスト
        """
        if not pages:
            return

        self.pages = pages

        #各ページの「検索対象テキスト」を組み立て
        #タイトル・説明・キーワードに重みをつけるため、文字列を繰り返す
        corpus = []
        for p in pages:
            #keywordsがカンマ区切りの文字列の場合はリストに追加
            kw = p.get("keywords", "") or ""
            if isinstance(kw, str):
                kw_list = [k.strip() for k in kw.split(",") if k.strip()]
            else:
                kw_list = kw

            #重みづけを実施。タイトルは3倍、説明は2倍、キーワードは2倍の重み
            text = " ".join([
                (p.get("title","") + " ") * 3,          #タイトルは3倍
                (p.get("description","") + " ") * 2,    #説明は2倍
                (p.get("full_text","") + " ") * 1,      #本文は1倍
                (" ".join(kw_list) + " ") * 2,          #キーワードは2倍
            ])
            corpus.append(text)

        #YF-IDFマトリックスを構築
        self.tfidf_matrix = self.vectorizer.fit_transform(corpus)
        self.is_fitted = True
        print(f"インデックス構築完了 :{len(pages)}ページ")

    def search(self,query: str,top_n: int = 20) -> list:
        """
        TF-IDFベースの検索を実行
        
        Args:
            query : 検索クエリ
            top_n : 返す結果の最大値
        
        Returns:
            スコア付きの検索結果リスト
        """
        if not self.is_fitted or not query.strip():
            return[]

        #クエリをベクトル化してコサイン炊事どを計算する（Step3で学んだ内容）
        query_vec = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vec,self.tfidf_matrix)[0]

        #閾値以上のページだけ結果に含める
        results = []
        for idx, base_score in enumerate(similarities):
            if base_score > 0.01:
                page = self.pages[idx].copy()

                #追加スコアりんぐで最終スコアを計算
                final_score = self._calculate_final_score(page, base_score, query)

                #スコアをパーセント表示用に変換する
                page["relevance_score"] = round(float(final_score) *100, 1)
                page["base_score"] = round(float(base_score) * 100, 1)
                results.append(page)

        #スコアの高い順に並べて　top_n 件を返す
        results.sort(key=lambda x: x["relevance_score"],reverse=True)
        return results[:top_n]

    def _calculate_final_score(self, page: dict, base_score: float,query: str) -> float:
        """
        複数の要素を組み合わせて最終スコアを計算する（内部メソッド）
        Args：
            page: ページ情報
            base_score: TF-IDFベーススコア
            query: 検索クエリ
        
        Returns:
            最終スコア
        """

        score = base_score
        query_lower = query.lower()

        #1.タイトルマッチボーナス
        title = page.get("title", "").lower()
        if query_lower == title:
            score *=  1.8       #完全一致の場合、＋80%にする
        elif query_lower in title:
            score *= 1.4    #部分一致の場合、＋40%

        #2.キーワードマッチボーナス
        keywords = page.get("keywords",[])

        if isinstance(keywords, str):
            keywords = keywords.split(",")

        keywords_lower = [k.strip().lower() for k in keywords]

        if query_lower in keywords_lower:
            score *= 1.3    #キーワード数：＋30%

        #3.新鮮度ボーナス（90日以内おページは最大　＋20%）
        crawled_at = page.get("crawled_at","")
        if crawled_at:
            try:
                crawled = datetime.fromisoformat(crawled_at.replace("Z","+00:00"))
                days_old = (datetime.now() - crawled.replace(tzinfo=None)).days
                if days_old <= 90:
                    recency_bonus = 1 + (0.2 * (90 - days_old) / 90)
                    score *= recency_bonus
            except Exception:
                pass

        #4.文字数による調査
        word_count = page.get("word_count" ,0)
        if word_count < 50:
            score *= 0.7        #短すぎるページは70%に減点
        elif word_count > 10000:
            score *= 0.85       #長すぎるページは85%に減点

        return score

# ==================================================
# 動作確認（追加課題の処理）
# ==================================================

# 1. 検索エンジンの初期化とインデックス構築
engine = SearchEngine()
engine.build_index(sample_pages)


# --------------------------------------------------
# 課題①：検索クエリを変えて比較する
# --------------------------------------------------
print("\n" + "="*50)
print("【課題①】検索クエリを変えてランキングを比較")
print("="*50)

queries = ["AI 機械学習", "IoT", "DX"]

for q in queries:
    results = engine.search(q)
    print(f"\nクエリ「{q}」→ {len(results)} 件ヒット")
    for n, r in enumerate(results, 1):
        # 順位、タイトル、最終スコア(relevance)、ベーススコア(base)を表示
        print(f"  {n}位  {r['title']:<20} relevance={r['relevance_score']:>6}%  base={r['base_score']:>6}%")


# --------------------------------------------------
# 課題②：タイトルボーナスを 1.4 -> 2.0 に変えた場合をシミュレーション
# --------------------------------------------------
print("\n" + "="*50)
print("【課題②】タイトルボーナスを 1.4 から 2.0 に変更した影響")
print("="*50)

BEFORE = 1.4  # 変更前の倍率
AFTER = 2.0   # 変更後の倍率

print(f"{'ページ名':<22}{'変更前スコア':>10}{'変更後スコア':>10}  変化")
print("-" * 55)

for r in engine.search("DX"):
    title = r.get("title", "").lower()
    before_score = r["relevance_score"]

    # タイトルに「dx」が含まれており、かつ「dx」そのものではない場合（部分一致）
    if "dx" in title and title != "dx":
        # 1.4で割って元の値に戻し、2.0を掛け直す
        after_score = round(before_score / BEFORE * AFTER, 1)
        comment = "← ボーナス増でスコア上昇！"
    else:
        after_score = before_score
        comment = "（タイトルに無いため変化なし）"

    print(f"{r['title'][:20]:<22}{before_score:>10}%{after_score:>10}%  {comment}")

インデックス構築完了 :3ページ

【課題①】検索クエリを変えてランキングを比較

クエリ「AI 機械学習」→ 1 件ヒット
  1位  AI活用ガイド              relevance=  34.9%  base=  49.9%

クエリ「IoT」→ 1 件ヒット
  1位  IoTセンサー活用事例          relevance=  77.2%  base=  42.4%

クエリ「DX」→ 2 件ヒット
  1位  DX戦略レポート2024         relevance=  48.0%  base=  26.4%
  2位  IoTセンサー活用事例          relevance=  27.4%  base=  21.1%

【課題②】タイトルボーナスを 1.4 から 2.0 に変更した影響
ページ名                      変更前スコア    変更後スコア  変化
-------------------------------------------------------
DX戦略レポート2024                48.0%      68.6%  ← ボーナス増でスコア上昇！
IoTセンサー活用事例                 27.4%      27.4%  （タイトルに無いため変化なし）


4-6.シングルトンインスタンスでSearchEngineを管理する
・Step4-5では、engine = SearchEngine()と直接インスタンスを作成した。
・しかしStreamlitアプリでは、毎回SearchEngineを作ると毎回インデックスを再構築してしまうという問題がある。

┗例え
・カレーを作る（野菜を切る、煮る、etc...）
・2回目以降は残りのカレーを冷凍庫に保存するので、1から作らなくていい。
・なので、毎回1から作らなくていい。
　┗この冷凍庫が、RAM
・アプリを終了すると冷凍庫が空になる

In [40]:
from typing import List

#シングルトンインスタンス（モジュールレベルで保持）
_engine = None
def get_engine() -> SearchEngine:
    """
    検索エンジンのシングルトンインスタンスを取得
    """
    global _engine
    if _engine is None:
        _engine = SearchEngine()
    return _engine

def rebuild_index(pages: List[dict]):
    """
    検索エンジンのインデックスを再構築する（新しいページが追加された時に呼び出す）
    """
    engine = get_engine()
    engine.build_index(pages)

#動作確認
engine2 = get_engine()
engine3 = get_engine()
print(f"同じインスタンス？{engine2 is engine3}")  # True が出力されるはず
print("→ get_engine()で取得したインスタンスは同じSearchEngineが返す")

同じインスタンス？True
→ get_engine()で取得したインスタンスは同じSearchEngineが返す


これらをつなぎ、ranking.pyを作る

基本的には、
`TfidfVectorizer` + `cosine_similarity` | ライブラリ版。高速・高精度 |
| `SearchEngine` クラス | `build_index()` → `search()` の本番仕様 |
この2つを掛け合わせる。
完成系は、以下のPython

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer #数値の表に変換する
from sklearn.metrics.pairwise import cosine_similarity #数値どうしを比べて、どのぐらい近いかを計算する
from datetime import datetime #日時を管理する（デフォルト入れておく）
from typing import List #List[src]のように型を指定するためのもの

class SearchEngine:
    """TF-IDFベースの検索エンジン（ranking.py)の本体を作る"""

    def __init__(self):
        #TF-IDFのベクトライザーを初期化
        #日本語は単語の間にスペースがないため、既存の「単語区切り」では
        #「営業店DX推進レポート」が丸ごと1語になり、「DX」で検索をかけても一致しない
        #そこで、analyzer = "char_wb" （文字N-gram）で2~3文字のまとまりを特徴量にする
        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb",     #文字N-gram（日本語のまま使えるようにする）
            ngram_range=(2,3),      #2~3文字のまとまり
            max_features=5000,
            min_df=1,
            max_df=0.95,
            sublinear_tf=True       #TFの対数スケーリング
        )
        self.tfidf_matrix = None    #インデックス（後で構築）
        self.pages= []              #元のページデータを保持
        self.is_fitted = False      #インデックスが構築済みがのフラグ

    def build_index(self, pages:list):
        """
        全ページのTF-IDFインデックスを構築

        Args:
            pages:ページ情報の辞書リスト
        """
        if not pages:
            return

        self.pages = pages

        #各ページの「検索対象テキスト」を組み立て
        #タイトル・説明・キーワードに重みをつけるため、文字列を繰り返す
        corpus = []
        for p in pages:
            #keywordsがカンマ区切りの文字列の場合はリストに追加
            kw = p.get("keywords", "") or ""
            if isinstance(kw, str):
                kw_list = [k.strip() for k in kw.split(",") if k.strip()]
            else:
                kw_list = kw

            #重みづけを実施。タイトルは3倍、説明は2倍、キーワードは2倍の重み
            text = " ".join([
                (p.get("title","") + " ") * 3,          #タイトルは3倍
                (p.get("description","") + " ") * 2,    #説明は2倍
                (p.get("full_text","") + " ") * 1,      #本文は1倍
                (" ".join(kw_list) + " ") * 2,          #キーワードは2倍
            ])
            corpus.append(text)

        #YF-IDFマトリックスを構築
        self.tfidf_matrix = self.vectorizer.fit_transform(corpus)
        self.is_fitted = True
        #print(f"インデックス構築完了 :{len(pages)}ページ") これは不要

    def search(self,query: str,top_n: int = 20) -> list:
        """
        TF-IDFベースの検索を実行
        
        Args:
            query : 検索クエリ
            top_n : 返す結果の最大値
        
        Returns:
            スコア付きの検索結果リスト
        """
        if not self.is_fitted or not query.strip():
            return[]

        #クエリをベクトル化してコサイン炊事どを計算する（Step3で学んだ内容）
        query_vec = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vec,self.tfidf_matrix)[0]

        #閾値以上のページだけ結果に含める
        results = []
        for idx, base_score in enumerate(similarities):
            if base_score > 0.01:
                page = self.pages[idx].copy()

                #追加スコアりんぐで最終スコアを計算
                final_score = self._calculate_final_score(page, base_score, query)

                #スコアをパーセント表示用に変換する
                page["relevance_score"] = round(float(final_score) *100, 1)
                page["base_score"] = round(float(base_score) * 100, 1)
                results.append(page)

        #スコアの高い順に並べて　top_n 件を返す
        results.sort(key=lambda x: x["relevance_score"],reverse=True)
        return results[:top_n]

    def _calculate_final_score(self, page: dict, base_score: float,query: str) -> float:
        """
        複数の要素を組み合わせて最終スコアを計算する（内部メソッド）
        Args：
            page: ページ情報
            base_score: TF-IDFベーススコア
            query: 検索クエリ
        
        Returns:
            最終スコア
        """

        score = base_score
        query_lower = query.lower()

        #1.タイトルマッチボーナス
        title = page.get("title", "").lower()
        if query_lower == title:
            score *=  1.8       #完全一致の場合、＋80%にする
        elif query_lower in title:
            score *= 1.4    #部分一致の場合、＋40%

        #2.キーワードマッチボーナス
        keywords = page.get("keywords",[])

        if isinstance(keywords, str):
            keywords = keywords.split(",")

        keywords_lower = [k.strip().lower() for k in keywords]

        if query_lower in keywords_lower:
            score *= 1.3    #キーワード数：＋30%

        #3.新鮮度ボーナス（90日以内おページは最大　＋20%）
        crawled_at = page.get("crawled_at","")
        if crawled_at:
            try:
                crawled = datetime.fromisoformat(crawled_at.replace("Z","+00:00"))
                days_old = (datetime.now() - crawled.replace(tzinfo=None)).days
                if days_old <= 90:
                    recency_bonus = 1 + (0.2 * (90 - days_old) / 90)
                    score *= recency_bonus
            except Exception:
                pass

        #4.文字数による調査
        word_count = page.get("word_count" ,0)
        if word_count < 50:
            score *= 0.7        #短すぎるページは70%に減点
        elif word_count > 10000:
            score *= 0.85       #長すぎるページは85%に減点

        return score


"""── シングルトン（アプリ全体で1つのエンジンを使いまわす）"""
_engine = None
def get_engine() -> SearchEngine:
    """
    検索エンジンのシングルトンインスタンスを取得
    """
    global _engine
    if _engine is None:
        _engine = SearchEngine()
    return _engine

def rebuild_index(pages: List[dict]):
    """
    検索エンジンのインデックスを再構築する（新しいページが追加された時に呼び出す）
    """
    engine = get_engine()
    engine.build_index(pages)

Step5: app.py V1.0に統合する
以下、3週間分の部品をapp.V1.0で繋ぐ。
 ┗ W2のcrawler.py
 ┗ W3のdatabase.py
 ┗ W4のranking.py



In [41]:
from database import init_db, migrate_from_json, get_all_pages

init_db()  # DB（tech0_search.db）を自動作成
n = migrate_from_json("pages_w2.json")  # JSONからデータを移植
print(f"{n} 件を移行しました")
print(f"DBの登録件数：{len(get_all_pages())} 件")

25 件を移行しました
DBの登録件数：26 件


In [1]:
from database import init_db, migrate_from_json, get_all_pages

init_db()                             # 空のDBを作成
n = migrate_from_json("pages_w2.json")  # JSONからデータを移植
print(f"{n} 件を移行しました")

25 件を移行しました


App.pyにモジュールを統合する

Step 4 で **検索エンジン（ranking.py）** を、W3 で **DB操作（database.py）** を作った。
ここでは、それらの部品を **app.py の中で使えるようにつなぐ** 方法を学ぶ。

```
Step 4 で作った → ranking.py（SearchEngine クラス）
W3 で作った   → database.py（DB の読み書き）
W2 から流用  → crawler.py（Web クロール）
                       ↓
               app.py で全部つなぐ！


                💡 **料理キットで考えると**
>
> `database.py`（冷蔵庫係）が食材を管理し、
> `crawler.py`（買い物係）が新しい食材を仕入れ、
> `ranking.py`（シェフ）が料理（= 検索結果）を作る。
> `app.py` はそれを **お客さん（ユーザー）に出すホールスタッフ** だ。

### 〈② import の書き方〉

別の `.py` ファイルの関数を使いたいときは **`import`** を使う。

**書き方は2種類ある：**

```python
# パターン①：モジュールごと読み込む
import database
database.init_db()  # モジュール名.関数名() で呼び出す

# パターン②：使いたい関数だけ読み込む（今回はこちら）
from database import init_db, get_all_pages
init_db()           # 関数名だけで呼び出せる（短くて読みやすい）

In [2]:
# import が正しく動作するか確認しよう
#
# ⚠️ このセルは「同じフォルダに database.py / ranking.py / crawler.py がある」ときだけ成功します。
#    このノートブックの場所で実行すると ModuleNotFoundError になりますが、
#    それは失敗ではなく「import の仕組みを確かめられた」ということです。



try:
    from database import init_db, get_all_pages, insert_page, log_search
    from ranking  import get_engine, rebuild_index
    from crawler  import crawl_url

    print("✅ 3つのモジュールを読み込めました")
    print()
    print("database.py から：")
    print(f"   init_db       → {init_db}")
    print(f"   get_all_pages → {get_all_pages}")
    print(f"   insert_page   → {insert_page}")
    print("ranking.py から：")
    print(f"   get_engine    → {get_engine}")
    print(f"   rebuild_index → {rebuild_index}")
    print("crawler.py から：")
    print(f"   crawl_url     → {crawl_url}")
    print()
    print("→ 統合の準備完了！")

except ModuleNotFoundError as e:
    print(f"❌ 読み込めませんでした： {e}")
    print()
    print("── これは想定どおりの結果です ──")
    print("import は「同じフォルダにあるファイル」を探しにいきます。")
    print("いまこのノートブックがある場所には、まだ3つの .py がありません。")
    print()
    print("試したい人は、次のどちらかで実行してみてください：")
    print("  ① 自分で作った tech0-search-v1.0/ フォルダにこのコードをコピーする")
    print("  ② answers/w4/ フォルダにコピーする（完成コードが置いてあります）")


✅ 3つのモジュールを読み込めました

database.py から：
   init_db       → <function init_db at 0x1042c40e0>
   get_all_pages → <function get_all_pages at 0x1042c42c0>
   insert_page   → <function insert_page at 0x1042c4220>
ranking.py から：
   get_engine    → <function get_engine at 0x1042c44a0>
   rebuild_index → <function rebuild_index at 0x1388934c0>
crawler.py から：
   crawl_url     → <function crawl_url at 0x138e6aa20>

→ 統合の準備完了！


In [3]:
# ── 演習 5-A 解答欄 ──
# insert_page を使うための import 文を書いてみよう

# ここに書く ↓
# from _______ import _______
from database import insert_page

# ── 書けたか確認 ──
try:
    insert_page
    print("✅ insert_page を読み込めています")
except NameError:
    print("→ まだ import できていません。上に import 文を書いてみましょう。")
except ModuleNotFoundError as e:
    print(f"→ {e}")
    print("  （このノートブックの場所には database.py が無いため、これは想定どおりです）")

✅ insert_page を読み込めています
